## 04. scratchpad

### 1. Data Upload

In [1]:
import pandas as pd
import os
from collections import defaultdict, Counter
import re

In [2]:
def upload_to_dataframe(root, files, num_lines):
    path_eng, path_pol = [root+f for f in files]
    data = defaultdict(list)
    
    with open (path_eng, encoding="utf_8") as f_eng, open (path_pol, encoding="utf_8") as f_pol:
        for _ in range(num_lines):
            data['eng_text'].append(f_eng.readline().strip())
            data['pol_text'].append(f_pol.readline().strip())
    return pd.DataFrame(data)

In [3]:
root = "../data/opus_opensub/en-pl.txt/"
files = ('OpenSubtitles.en-pl.en', '/OpenSubtitles.en-pl.pl')
df = upload_to_dataframe(root, files, 600000)

### 2. EDA & Sanitazation

#### 2.1 English Non-Ascii Sentences

In [4]:
def nonascii_list(df_series, is_pol):
    if is_pol:
        pl_chars = set("ąęćłńóśźżĄĘĆŁŃÓŚŹŻ")
        requir = lambda x: x.isascii() or x in pl_chars
    else:
        requir = lambda x: x.isascii()
        
    text_all = ''.join(df_series)
    unique_dct = {x for x in text_all if not requir(x)}
    return sorted(list(unique_dct))

In [5]:
print(nonascii_list(df['eng_text'], False))

['\x80', '\x81', '\x82', '\x83', '\x84', '\x85', '\x88', '\x8b', '\x8c', '\x8e', '\x8f', '\x91', '\x94', '\x96', '\x98', '\x99', '\x9c', '\x9d', '\xa0', '¡', '¢', '£', '¤', '¥', '¦', '§', '¨', '©', 'ª', '¬', '\xad', '°', '±', '²', '³', '´', 'µ', '¶', '·', '¸', '¹', 'º', '¼', '½', '¾', '¿', 'Á', 'Â', 'Ã', 'Ä', 'Å', 'Ç', 'É', 'Ê', 'Ì', 'Ï', 'Ð', 'Ñ', 'Ô', 'Ö', 'Ø', 'Ü', 'Ý', 'Þ', 'ß', 'à', 'á', 'â', 'ã', 'ä', 'å', 'ç', 'è', 'é', 'ê', 'ì', 'í', 'î', 'ï', 'ð', 'ñ', 'ó', 'ô', 'ö', 'ù', 'ú', 'û', 'ü', 'ý', 'þ', 'Ă', 'ď', 'ę', 'Ł', 'ń', 'ő', 'œ', 'ś', 'Š', 'š', 'Ј', 'С', 'й', '،', 'أ', 'إ', 'ئ', 'ا', 'ب', 'ة', 'ت', 'ح', 'ر', 'س', 'ش', 'ع', 'غ', 'ف', 'ك', 'ل', 'م', 'ن', 'و', 'ي', 'َ', 'ُ', 'ِ', 'ّ', 'ْ', '\u200b', '‒', '–', '—', '‚', '€', '™', '─', '☻', '♥', '♪', '慹', '拢', '檛', '鈥', 'ﬁ', 'ﬂ']


In [6]:
is_ascii = lambda text: text.isascii()
maska = df['eng_text'].apply(is_ascii)
df_non_ascii = df[~maska].copy().reset_index(drop=True)
print(df_non_ascii.shape)
df_non_ascii.head()

(4595, 2)


,eng_text,pol_text
0,¿You had never seen A crab suedes? Them there ...,- Nigdy wcześniej nie widziałaś raków?
1,"He hears ¿, where you live?",Nigdy wcześniej nie byłam w lesie.
2,"Boys ¿, they are there?",Jesteście tam chłopaki?
3,- Te atrapé. - No es gracioso. Your he catches...,- Ale twoje mina była.
4,"Your also ¿, it is not like that?","- Tak, niezła jest"


In [7]:
df_2 = df[maska].reset_index(drop=True)
df_2.shape

(595405, 2)

#### 2.2 Polish Non-Ascii Sentences

In [8]:
pl_chars = set("ąęćłńóśźżĄĘĆŁŃÓŚŹŻ")
requir = lambda let: not let.isascii() and let not in pl_chars
maska = df_2['pol_text'].apply(lambda snt: any([requir(let) for let in snt]))
df_non_ascii = df_2[maska].copy().reset_index(drop=True)
print(df_non_ascii.shape)
df_non_ascii.sample(10)

(1227, 2)


,eng_text,pol_text
1138,"In June, July and August We look cute when we'...","¶ Czerwiec, lipiec, sierpień ¶ W szortach cudn..."
463,- Well...,- Cóž...
666,- I've had to compromise.,Ale musialem išc na kompromis.
647,Smart enough to understand a situation when it...,"Jest pan na tyle madry, by pojac nasze wyjašni..."
693,"Well, I... see by the papers you certainly got...","Widze z gazet, že stales sie prawdziwym senato..."
968,"Calm down, darling.","Rémy! - Przestań, skarbie."
238,- We can't find him.,–Bądź co bądź nie możemy go znaleźć.
712,- Clarissa... where can we get a drink?,- Clarisso... gdzie možemy sie napic?
866,"- Oh, fine. Fine, Joe.","- Šwietnie, Joe."
364,Garcon!,- Garçon!


In [9]:
df_3 = df_2[~maska].reset_index(drop=True)
print(df_3.shape)
df_3.sample(5)

(594178, 2)


,eng_text,pol_text
582214,Know what happened to her?,Gdzie ona teraz jest?
145620,"What's he saying, Bill?","Co on mówi, Bill?"
92851,Ya can't argue against that.,Nie możesz temu zaprzeczać.
374706,I've done everything in my power to help you f...,"Zrobiłem wszystko, co w mojej mocy, aby pomóc ..."
71572,- I don't care for the value.,- Nie interesuje mnie to.


In [10]:
print(nonascii_list(df_3['eng_text'], False))
print(nonascii_list(df_3['pol_text'], True))

[]
[]


In [11]:
df_3.iloc[74182]['eng_text']

'The next time I...'

In [12]:
df_3.iloc[74182]['pol_text']

'Następnym razem...'

#### 2.3 Short Or Empty Examples

In [13]:
mask1 = df_3['eng_text'].str.len() >= 3
mask2 = df_3['pol_text'].str.len() >= 3

In [14]:
df_4 = df_3[mask1 & mask2].reset_index(drop=True)
df_4.head()

,eng_text,pol_text
0,"Previously on ""The Blacklist""...",/W poprzednich odcinkach: /
1,- You want to call your daddy?,- Chcesz zadzwonić do taty?
2,"- Yeah, I want to tell him I'm okay.","- Tak, powiem, że wszystko w porządku."
3,Okay.,Dobrze.
4,Lizzy... Be careful of your husband.,"Lizzy, uważaj na męża."


In [15]:
df_4.shape

(593894, 2)

#### 2.4 Dialog like sentences [starts with - ]

In [16]:
is_dialog = lambda snt: bool(re.match(r"^ *- *", snt))
mask = df_4['eng_text'].apply(is_dialog) | df_4['pol_text'].apply(is_dialog)
df_dialogs = df_4[mask].copy().reset_index(drop=True)
df_dialogs.head()

,eng_text,pol_text
0,- You want to call your daddy?,- Chcesz zadzwonić do taty?
1,"- Yeah, I want to tell him I'm okay.","- Tak, powiem, że wszystko w porządku."
2,- 45 seconds.,Szybciej! 45 sekund.
3,It's okay. It's okay. It's okay.,- Już dobrze.
4,It's okay. You're okay. You're okay.,- Odpoczywaj.


In [17]:
def fix_dialog(snt):
    return re.sub(r"^ *- *", "", snt)

df_4['eng_text'] = df_4['eng_text'].apply(fix_dialog).reset_index(drop=True)
df_4['pol_text'] = df_4['pol_text'].apply(fix_dialog).reset_index(drop=True)
df_4.sample(30)

,eng_text,pol_text
496234,I hear Karl-Magnus has returned home.,"Słyszę, że Karl-Magnus wrócił do domu."
111803,"Why, I've been at...",Byłem u...
562568,You must look after her.,Musi się nią zająć.
435316,But-,Ale...
392345,We'll split the difference.,Damy na środku.
36373,"This fire, this voice, this agony?","Ogniowi, głosowi, męczarni."
189920,"It's nice to see you, Lehmann.","Miło cię widzieć, Lehmann."
195345,Why not? I guess we can make you talk.,"Myślę, że możemy zmusić cię do mówienia."
428551,Haven't I told you not to answer the phone at ...,"Nie mówiłem ci, żeby w nocy nie odbierać telef..."
447138,Thank you.,"Możesz iść. Dziękuję, siostro."


#### 2.5 Broken Dialog like sentences [somehwere has -]

# Do naprawy (lepszy re przed tokenziacja)

In [18]:
broken_dialog = lambda snt: bool(re.match(r".* +- +.*", snt))
mask1 = df_4['eng_text'].apply(broken_dialog)
mask2 = df_4['pol_text'].apply(broken_dialog)

In [19]:
df_4[mask1 | mask2].sample(5)

,eng_text,pol_text
326393,You had a gang of what?,Jaki gang? - Chłopaków.
361743,When Father set up the trust for the boys why ...,Dlaczego ojciec założył fundusz powierniczy dl...
7083,"""For the first time in my life I am frightened...",Ostrzegam cię - uważaj na tego Wenk'a
510537,"Can I get you something else? - Nothing more, ...",moja droga.
87346,Good. - Are you coming in?,Zejdziecie?


In [20]:
df_5 = df_4[~mask1 & ~mask2].reset_index(drop=True)

In [21]:
broken_dialog = lambda snt: bool(re.match(r".*-.*", snt))

In [22]:
df_5.sample(50)

,eng_text,pol_text
183602,"What do I care? Alright boys, put him in.","Dobra, weźcie go tam."
355135,"Goodbye, Uncle Charles.","Do widzenia, wujku."
268757,I'm trying to concentrate.,Pozwól się skupić!
349750,I shall see that you're not disturbed sir.,"Dopilnuję, żeby panu nie przeszkadzano."
361649,There's somebody down here to see you.,Ktoś chce się z wami zobaczyć.
544787,Let him have a drink!,Niech się napije!
356193,My brother said later!,"Mój brat powiedział, że później!"
569136,I pressed the gun.,Chwyciłam broń.
177389,Door shaker!,Szejker drzwi!
449319,I didn't want to believe it.,Nie chciałem uwierzyć.


#### 2.6 Different Last Case

In [23]:
mask = df_5['eng_text'].str[-1] != df_5['pol_text'].str[-1]
mask.sum()

57558

In [24]:
df_6 = df_5[~mask].reset_index(drop=True)

#### 2.7 Text length difference

In [25]:
mask = df_6['eng_text'].str.split
df_6['len_ratio'] = df_6['eng_text'].str.split().str.len() / df_6['pol_text'].str.split().str.len()
df_6.head()

,eng_text,pol_text,len_ratio
0,You want to call your daddy?,Chcesz zadzwonić do taty?,1.500000
1,"Yeah, I want to tell him I'm okay.","Tak, powiem, że wszystko w porządku.",1.333333
2,Okay.,Dobrze.,1.000000
3,Lizzy... Be careful of your husband.,"Lizzy, uważaj na męża.",1.500000
4,I can only lead you to the truth.,/Mogę naprowadzić cię na prawdę.,1.600000


In [26]:
thres_left = df_6["len_ratio"].quantile(0.05)
thres_right = df_6["len_ratio"].quantile(0.95)
thres_left, thres_right

(0.75, 2.5)

In [27]:
mask = (df_6["len_ratio"] >= thres_left) & (df_6["len_ratio"] <= thres_right)
mask.sum()

472209

In [28]:
df_7 = df_6[mask].reset_index(drop=True)
df_7.sample(20)

,eng_text,pol_text,len_ratio
358124,lt's very kind of you.,To miłe z Pana strony.,1.000000
448833,I'm going to write him tonight.,Napiszę do niego dziś wieczorem.,1.200000
159532,Pretty enough for Ted's grave.,Jak znalazł na grób Teda.,1.000000
342397,"If the mist clears, we're cooked!","Jeśli mgły opadną, koniec z nami!",1.000000
310469,I was hoping you would.,Taką miałem nadzieję.,1.666667
378740,Do you know anything about building?,Czy wiesz coś na temat budowania?,1.000000
131365,"Well, will Dan go, too?",Czy Dan też wypływa?,1.250000
425898,No.,Nie.,1.000000
265153,I was imagining the criminal was clever enough...,"Wyobrażałam sobie, że przestępca jest na tyle ...",1.000000
112639,We're waiting till General Merritt gets here w...,"Czekamy, aż generał Merritt przybędzie tu z Pi...",1.214286


In [29]:
df_7.sample(50)

,eng_text,pol_text,len_ratio
417593,I'd turn them into Army ordnance.,Słyszałem od artylerzystów.,2.000000
67909,Then shut up about it.,Więc o tym nie wspominaj.,1.000000
145466,"You know, this is a beautiful spot here, Hartley.","Wiesz, że to piękne miejsce, Hartley.",1.500000
52875,You can Steal a picture.,Można ukraść zdjęcie.,1.666667
80180,"Becky, Becky, don't... I...","Becky, Becky przestań, ja...",1.000000
78128,had taken his own life.,który odebrał sobie życie.,1.250000
2158,What are you up to?,Co zamierzasz?,2.500000
347344,"That's one of them, the other one's in the cel...",To jedno z nich. Drugie jest w piwnicy.,1.250000
270425,"Never fear, my little one.","Nie martw się, moja mała.",1.000000
149243,These girls know too much.,Te dziewczyny wiedzą za dużo.,1.000000


#### 2.8 Sentences starting with * fix

In [30]:
fix_star = lambda snt: re.sub(r"^ *[*].*", "", snt)
df_7['eng_text'] = df_7['eng_text'].apply(fix_star)
df_7['pol_text'] = df_7['pol_text'].apply(fix_star)

#### 2.9 Action Comments

In [31]:
def mask_distinct(df, col1, col2, chars):
    chars_escaped = "".join(re.escape(c) for c in chars)
    templ = f"[{chars_escaped}]"

    m1 = df[col1].str.contains(templ, regex=True, na=False)
    m2 = df[col2].str.contains(templ, regex=True, na=False)

    return m1 ^ m2

chars = '[]()/\*+#$"️'
mask = mask_distinct(df_7, "eng_text", "pol_text", chars)

df_8 = df_7[~mask].reset_index(drop=True)
df_8.shape

(467550, 3)

In [32]:
df_8.sample(50)

,eng_text,pol_text,len_ratio
166472,Maybe he's handsome like Ed.,Moze on jest przystojny jak Ed.,0.833333
396776,"Don't you find Rio a little hard to take, too?","Nie uważasz, że ciężko jest w Rio?",1.428571
465149,Are you asking me?,Prosisz mnie?,2.000000
78462,Please accept my apologies.,Proszę przyjąć moje przeprosiny.,1.000000
252294,I sort of liked it there.,I tak jakby spodobało mi się.,1.000000
386642,I see.,Rozumiem.,2.000000
294874,Even better than I was before.,Nawet lepiej niż przedtem.,1.500000
341085,I didn't tell them what you were doing.,"Nie o tym, co robisz.",1.600000
86776,All men to your posts!,Wszyscy na stanowiska!,1.666667
92908,"Norton, get me a cup.","Panie Norton, poproszę o kubek.",1.000000


#### 2.10  " ` " ---> " ' " & del " ~ "

In [33]:
wanted_case = '`'
mask = df_8["eng_text"].apply(lambda x: wanted_case in x)
df_8[mask]

,eng_text,pol_text,len_ratio
214145,"Revolution had broken out, her diplomats sued ...","Rewolucja na tyłach frontu, dąży do zawarcia p...",1.277778
214152,We`ll examine it.,Zbadajmy go.,1.500000
214157,What do you think you`re doing?,Wstawaj! Co ty robisz?,1.500000
214161,Where`s your hand grenade?,Gdzie masz swój granat?,1.000000
214163,Let `em have it!,Niech je poczują!,1.333333
...,...,...,...
214943,We don`t want to hate one another.,Bez nienawiści ani pogardy.,1.750000
214944,There`s room for everyone.,Każdy ma swe miejsce.,1.000000
214947,"Greed has poisoned men`s souls, has barricaded...","Chciwość zatruła nasze dusze, wzniosła mury ni...",1.333333
214963,You don`t hate.,Przestańcie nienawidzieć.,1.500000


In [34]:
df_8["eng_text"] = df_8["eng_text"].str.replace("`", "'", regex=False)
df_8["pol_text"] = df_8["pol_text"].str.replace("`", "'", regex=False)

In [35]:
df_8["eng_text"] = df_8["eng_text"].str.replace("~", "", regex=False)
df_8["pol_text"] = df_8["pol_text"].str.replace("~", "", regex=False)

#### 2.11 Del rows with specific case

In [36]:
def del_row_with_char(df, eng_col, pol_col, chars):
    df_proc = df.copy()
    for char in chars:
        mask1 = df_proc[eng_col].apply(lambda x: char in x)
        mask2 = df_proc[pol_col].apply(lambda x: char in x)
        df_proc = df_proc[~(mask1 | mask2)].reset_index(drop=True)
    return df_proc
    

chars = "=+*@#;|_}{"
df_9 = del_row_with_char(df_8, "eng_text", "pol_text", chars)

#### 2.12 Del rows with dialog -

In [37]:
mask1 = df_9["eng_text"].apply(lambda x: bool(re.search(r' +-[^ ]+', x)))
mask2 = df_9["pol_text"].apply(lambda x: bool(re.search(r' +-[^ ]+', x)))
df_data = df_9[~(mask1 | mask2)].reset_index(drop=True)

#### 3. Data Tokenization

In [38]:
uniq_cases = set("".join(df_data["eng_text"].astype(str)))
tab1 = sorted(list(uniq_cases), reverse=True)
print(tab1)

['z', 'y', 'x', 'w', 'v', 'u', 't', 's', 'r', 'q', 'p', 'o', 'n', 'm', 'l', 'k', 'j', 'i', 'h', 'g', 'f', 'e', 'd', 'c', 'b', 'a', ']', '[', 'Z', 'Y', 'X', 'W', 'V', 'U', 'T', 'S', 'R', 'Q', 'P', 'O', 'N', 'M', 'L', 'K', 'J', 'I', 'H', 'G', 'F', 'E', 'D', 'C', 'B', 'A', '?', ':', '9', '8', '7', '6', '5', '4', '3', '2', '1', '0', '/', '.', '-', ',', ')', '(', "'", '%', '$', '"', '!', ' ']


In [39]:
uniq_cases = set("".join(df_data["pol_text"].astype(str)))
tab2 = sorted(list(uniq_cases), reverse=True)
print(tab2)

['ż', 'Ż', 'ź', 'Ź', 'ś', 'Ś', 'ń', 'Ń', 'ł', 'Ł', 'ę', 'Ę', 'ć', 'Ć', 'ą', 'Ą', 'ó', 'Ó', 'z', 'y', 'x', 'w', 'v', 'u', 't', 's', 'r', 'q', 'p', 'o', 'n', 'm', 'l', 'k', 'j', 'i', 'h', 'g', 'f', 'e', 'd', 'c', 'b', 'a', ']', '[', 'Z', 'Y', 'X', 'W', 'V', 'U', 'T', 'S', 'R', 'Q', 'P', 'O', 'N', 'M', 'L', 'K', 'J', 'I', 'H', 'G', 'F', 'E', 'D', 'C', 'B', 'A', '?', ':', '9', '8', '7', '6', '5', '4', '3', '2', '1', '0', '/', '.', '-', ',', ')', '(', "'", '%', '$', '"', '!', ' ']


In [40]:
def tokenize_eng(snt):
    snt = snt.lower()
    snt = re.sub("(?<! )'(?! )", r" '", snt)
    snt = re.sub(r"""([\]?:)\[(/\.\-,%\$"!])""", r" \1 ", snt)
    snt = re.sub(r"\s+", " ", snt).strip()
    return re.split(r'\s+', snt.strip())

In [41]:
def tokenize_pol(snt):
    snt = snt.lower()
    snt = re.sub(r"""([\]?:)\[(/\.\-,%\$"!])""", r" \1 ", snt)
    snt = re.sub(r"\s+", " ", snt).strip()
    return re.split(r'\s+', snt.strip())

In [42]:
df_data = df_data.drop(['len_ratio'], axis=1)

#### 3.1 Splitting and trimming on quantile

In [43]:
df_data['eng_split'] = df_data['eng_text'].apply(tokenize_eng)
df_data['pol_split'] = df_data['pol_text'].apply(tokenize_pol)
df_data.sample(10)

,eng_text,pol_text,eng_split,pol_split
301812,"Ah, ah, I'm okay.","Nie, nic mi nie jest.","[ah, ,, ah, ,, i, 'm, okay, .]","[nie, ,, nic, mi, nie, jest, .]"
131909,I came here to look at a man.,"Przyszłam tu, żeby spojrzeć na mężczyznę.","[i, came, here, to, look, at, a, man, .]","[przyszłam, tu, ,, żeby, spojrzeć, na, mężczyz..."
415710,"If Abrahams doesn't impress me, I'll try Van D...","Jeśli Abrahams mi się nie spodoba, zwrócę się ...","[if, abrahams, doesn, 't, impress, me, ,, i, '...","[jeśli, abrahams, mi, się, nie, spodoba, ,, zw..."
351137,"Monke, get some water.","Monke, przynieś wody.","[monke, ,, get, some, water, .]","[monke, ,, przynieś, wody, .]"
211568,Never pursue her.,Nigdy jej nie namawiaj.,"[never, pursue, her, .]","[nigdy, jej, nie, namawiaj, .]"
412837,Do you see Father?,Widzisz ojca?,"[do, you, see, father, ?]","[widzisz, ojca, ?]"
337502,"You, turn around.","Ty, odwróć się.","[you, ,, turn, around, .]","[ty, ,, odwróć, się, .]"
30430,HURRY UP.,Pospiesz się.,"[hurry, up, .]","[pospiesz, się, .]"
298697,Will you need me on the way back?,Potrzebny będę w drodze powrotnej?,"[will, you, need, me, on, the, way, back, ?]","[potrzebny, będę, w, drodze, powrotnej, ?]"
453763,"Inside, up the stairs.",Po schodach w górę.,"[inside, ,, up, the, stairs, .]","[po, schodach, w, górę, .]"


In [44]:
thres1 = df_data['eng_split'].str.len().quantile(0.99)
mask1 = df_data['eng_split'].str.len() > thres1

thres2 = df_data['pol_split'].str.len().quantile(0.99)
mask2 = df_data['pol_split'].str.len() > thres2
print(mask1.sum(), mask2.sum(), (mask1 | mask2).sum())

df_data = df_data[~(mask1 | mask2)].reset_index(drop=True)

4113 3960 5510


#### 3.2 BPE Init. Uniq_freq & Rozklad chars

In [148]:
uniq_freq_eng = Counter(df_data['eng_split'].explode())
uniq_freq_pol = Counter(df_data['pol_split'].explode())

In [149]:
len(uniq_freq_eng), len(uniq_freq_pol)

(40087, 113603)

In [250]:
char_factor = lambda word: [x for x in word] + ['</w>']
vocab_chars_eng = [[char_factor(k), v] for k, v in uniq_freq_eng.most_common()]
vocab_chars_pol = [[char_factor(k), v] for k, v in uniq_freq_pol.most_common()]
print(len(vocab_chars_eng), vocab_chars_eng[:5], '\n')
print(len(vocab_chars_pol), vocab_chars_pol[:5]) 

40087 [[['.', '</w>'], 435802], [[',', '</w>'], 177291], [['y', 'o', 'u', '</w>'], 134548], [['i', '</w>'], 122767], [['t', 'h', 'e', '</w>'], 89859]] 

113603 [[['.', '</w>'], 413780], [[',', '</w>'], 183948], [['?', '</w>'], 83444], [['n', 'i', 'e', '</w>'], 82694], [['t', 'o', '</w>'], 53190]]


#### 3.3 BPE Tokenizer most common pairs

In [249]:
class BPETokenizer():
    def __init__(self, vocab_chrs, vocab_size, is_tgt):
        self.vocab_chrs = vocab_chrs
        self.vocab_size = vocab_size
        self.vocab = self.vocab_init(vocab_chrs, is_tgt)
        self.base_size = len(self.vocab)

    def vocab_init(self, vocab_chrs, is_tgt):
        vocab = {'<pad>': 0, '<unk>': 1, '<eos>': 2, '</w>': 3}
        if is_tgt:
            vocab['<bos>'] = 4
            
        uniq_toks = {tok for toks, _ in vocab_chrs for tok in toks}
        uniq_toks.remove('</w>')
        for i, tok in enumerate(uniq_toks, len(vocab)):
            vocab[tok] = i
        return vocab                

    def train_bpe(self):
        for i in range(self.base_size, self.vocab_size):
            pair = self.most_common_pair()
            self.vocab[pair] = i
            pair_con = re.sub('\s+', '', pair)
            self.replace_pairs(pair, pair_con)

    def most_common_pair(self):
        pair_counts = Counter()
        for toks, freq in self.vocab_chrs:
            for pair in map(' '.join, zip(toks, toks[1:])):
                pair_counts[pair] += freq
        return pair_counts.most_common(1)[0][0]

    def replace_pairs(self, pair, pair_con):
        for i, (toks, _) in enumerate(self.vocab_chrs):
            self.vocab_chrs[i][0] = ' '.join(toks).replace(pair, pair_con).split()    

In [251]:
tokenizer_eng = BPETokenizer(vocab_chars_eng, 8000, False)

In [254]:
tokenizer_eng.train_bpe()

In [135]:
def connect_pairs(vocab_chars, pair):
    vocab_proc = vocab_chars.copy()
    pair_con = re.sub('\s+', '', pair)
    for i, (toks, _) in enumerate(vocab_proc):
        toks = " ".join(toks).replace(pair, pair_con).split()
        vocab_proc[i][0] = toks
    return vocab_proc

In [221]:
def most_common_pair(vocab_chars):
    pair_counts = Counter()
    for toks, freq in vocab_chars:
        for pair in map(' '.join, zip(toks, toks[1:])):
            pair_counts[pair] += freq
    return pair_counts.most_common(1)[0][0]

In [226]:
vocab_chars_eng

[[['.', '</w>'], 435802],
 [[',', '</w>'], 177291],
 [['y', 'o', 'u', '</w>'], 134548],
 [['i', '</w>'], 122767],
 [['t', 'h', 'e', '</w>'], 89859],
 [['?', '</w>'], 84640],
 [['t', 'o', '</w>'], 73192],
 [['a', '</w>'], 61661],
 [["'", 's', '</w>'], 57045],
 [['i', 't', '</w>'], 55074],
 [["'", 't', '</w>'], 50411],
 [['t', 'h', 'a', 't', '</w>'], 43020],
 [['a', 'n', 'd', '</w>'], 41124],
 [['o', 'f', '</w>'], 39514],
 [['i', 'n', '</w>'], 32142],
 [['!', '</w>'], 30794],
 [['m', 'e', '</w>'], 29977],
 [['w', 'h', 'a', 't', '</w>'], 26868],
 [['i', 's', '</w>'], 26378],
 [['h', 'e', '</w>'], 26149],
 [['f', 'o', 'r', '</w>'], 23935],
 [['w', 'e', '</w>'], 23370],
 [['t', 'h', 'i', 's', '</w>'], 20366],
 [['y', 'o', 'u', 'r', '</w>'], 20158],
 [['b', 'e', '</w>'], 20137],
 [['m', 'y', '</w>'], 20105],
 [["'", 'l', 'l', '</w>'], 20011],
 [['h', 'a', 'v', 'e', '</w>'], 19830],
 [['d', 'o', 'n', '</w>'], 18888],
 [['d', 'o', '</w>'], 18663],
 [['o', 'n', '</w>'], 18052],
 [['n', 'o', '</

In [102]:
def most_common_pair(vocab_chars):
    freq_pair = defaultdict(int)
    for toks, freq in vocab_chars:
        toks_pairs = [" ".join(pair) for pair in zip(toks, toks[1:])]
        for pair in toks_pairs:
            freq_pair[pair] += freq
    return max(freq_pair.items(), key=lambda x: x[1])[0]

In [190]:
pair_counts = Counter()
pair_counts[('t','h')] += 10
pair_counts[('t','h')] += 10
pair_counts.most_common(1)

[(('t', 'h'), 20)]

In [144]:
def bpe_tokenizer(vocab_chars, vocab_list, vocab_size):
    for _ in range(vocab_size):
        pair = most_common_pair(vocab_chars)
        vocab_list.append(re.sub('\s+', '', pair))
        print(pair)
        vocab_chars = connect_pairs(vocab_chars, pair)
        
    
    